# R1: Multi-Seed Evaluation — Seed 42

**Purpose:** Run RSQwen (Stage 1 + Stage 2) and Qwen Vanilla (Stage 1 only) for **seed=42**.

**Output:** `results_seed_42.json` + `predictions_seed_42.json` saved to Drive after **each model** completes.

**Estimated time:** ~3-5h on A100, ~5-7h on V100

**Run in parallel with:** the other two seed notebooks in separate Colab tabs.

## Step 1: Install Dependencies

In [1]:
!pip uninstall -y protobuf
!pip install -q protobuf==3.20.0
!pip install -q transformers datasets accelerate sentencepiece
!pip install -q scikit-learn pandas numpy tqdm openpyxl
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes
!pip install -q peft

import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['GRPC_VERBOSITY'] = 'ERROR'
warnings.filterwarnings('ignore')
print('Dependencies installed')


Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 16.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-monitoring 2.30.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.0 which is incompatible.
google-cloud-bigquery-connection 1.21.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.0 which is incompatible.
google-cloud-translate 3.26.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.0 which is incompatible.
googleapis-common-protos 1.74.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.0 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 3.20.0 which is incompatible.
wandb 0.25.1 requires protobuf!=5.28.0,!=5.29.0,<7,>4.21.

## Step 2: Mount Google Drive & Setup Paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, sys, os
from pathlib import Path

# === CONFIGURE THIS ===
DRIVE_BASE = Path('/content/drive/MyDrive/ViTHSD')
# ======================

WORK_DIR = Path('/content/vithsd')
WORK_DIR.mkdir(exist_ok=True)

for f in ['config.py', 'data_preparation.py', 'evaluation.py', 'models.py']:
    src = DRIVE_BASE / 'src' / f
    dst = WORK_DIR / f
    if src.exists():
        shutil.copy2(src, dst)
        print(f'  Copied {f}')
    else:
        raise FileNotFoundError(f'Missing: {src}')

os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))

OUTPUT_DIR = DRIVE_BASE / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Working dir: {WORK_DIR}')
print(f'Output dir: {OUTPUT_DIR}')


Mounted at /content/drive
  Copied config.py
  Copied data_preparation.py
  Copied evaluation.py
  Copied models.py
Working dir: /content/vithsd
Output dir: /content/drive/MyDrive/ViTHSD/outputs


## Step 3: Patch config.py paths for Colab

In [3]:
import config
config.DATA_DIR = DRIVE_BASE / 'data'
config.OUTPUT_DIR = OUTPUT_DIR
config.BASE_DIR = WORK_DIR

for f in ['train.xlsx', 'test.xlsx']:
    p = config.DATA_DIR / f
    assert p.exists(), f'Missing data file: {p}'
    print(f'  Found: {f}')

print('Config patched for Colab')


  Found: train.xlsx
  Found: test.xlsx
Config patched for Colab


## Step 4: GPU Detection

In [5]:
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > GPU')

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')

if gpu_mem >= 35:
    BATCH_SIZE = 8
elif gpu_mem >= 14:
    BATCH_SIZE = 4
else:
    BATCH_SIZE = 2

print(f'Batch size: {BATCH_SIZE}')


PyTorch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-80GB (85.1 GB)
Batch size: 8


## Step 5: Import Modules & Load Data

In [6]:
from config import FINAL_LABELS
from data_preparation import load_dataset_A
from evaluation import Evaluator
from models import create_model

train_texts, train_labels = load_dataset_A('train', data_dir=config.DATA_DIR)
test_texts, test_labels = load_dataset_A('test', data_dir=config.DATA_DIR)

print(f'Train: {len(train_texts)} samples')
print(f'Test: {len(test_texts)} samples')
print(f'Labels: {len(FINAL_LABELS)}')


[ViHSDLoader] Loaded 7000 samples from train
[ViHSDLoader] Loaded 1800 samples from test
Train: 7000 samples
Test: 1800 samples
Labels: 11


## Step 6: Load Rationale Data (for RSQwen Stage 2)

In [7]:
import json

rationale_path = DRIVE_BASE / 'data' / 'dataset_rationale.json'
assert rationale_path.exists(), f'Missing: {rationale_path}'

with open(rationale_path, 'r', encoding='utf-8') as f:
    rationale_data = json.load(f)

r_train_texts, r_train_implied, r_train_rationale, r_train_labels = [], [], [], []
for r in rationale_data:
    if str(r.get('dataset', '')).lower().strip() != 'train':
        continue
    content = (r.get('content') or '').strip()
    implied = (r.get('implied_statement') or '').strip()
    if not content or not implied:
        continue
    r_train_texts.append(content)
    r_train_implied.append(implied)
    r_train_rationale.append(r.get('rationale', []))
    r_train_labels.append(r.get('labels', []))

alignment_pairs = [{'content': c, 'implied': i} for c, i in zip(r_train_texts, r_train_implied)]

print(f'Rationale train samples: {len(r_train_texts)}')
print(f'Alignment pairs: {len(alignment_pairs)}')


Rationale train samples: 1221
Alignment pairs: 1221


## Step 7: Define Helper Functions

In [8]:
import random
import numpy as np
import torch
import gc
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f'  Seed set to {seed}')


def train_stage2(model, pairs, rationale_list, labels_list, num_epochs=1, learning_rate=5e-5):
    print(f'  Stage 2: {len(pairs)} pairs, {num_epochs} epoch(s), lr={learning_rate}')
    training_texts = []
    for i, pair in enumerate(pairs):
        content = pair['content']
        implied = pair['implied']
        rat = rationale_list[i] if i < len(rationale_list) else []
        lab = labels_list[i] if i < len(labels_list) else ['normal']
        input_text = model._format_input_cot(content)
        output_text = model._format_output_cot(lab, rat, implied)
        training_texts.append(input_text + output_text)

    class TextDataset(Dataset):
        def __init__(self, texts, tokenizer, max_length):
            self.encodings = tokenizer(
                texts, truncation=True, padding=True,
                max_length=max_length, return_tensors='pt'
            )
        def __len__(self):
            return len(self.encodings['input_ids'])
        def __getitem__(self, idx):
            return {
                'input_ids': self.encodings['input_ids'][idx],
                'attention_mask': self.encodings['attention_mask'][idx]
            }

    dataset = TextDataset(training_texts, model.tokenizer, model.max_length)
    dataloader = DataLoader(dataset, batch_size=model.batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.model.parameters(), lr=learning_rate)

    model.model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        progress = tqdm(dataloader, desc=f'  Stage2 Epoch {epoch+1}/{num_epochs}')
        for batch in progress:
            input_ids = batch['input_ids'].to(model.device)
            attention_mask = batch['attention_mask'].to(model.device)
            optimizer.zero_grad()
            outputs = model.model(
                input_ids=input_ids, attention_mask=attention_mask, labels=input_ids
            )
            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.model.parameters(), 1.0)
            optimizer.step()
            progress.set_postfix({'loss': f'{loss.item():.4f}'})
        avg_loss = total_loss / len(dataloader)
        print(f'  Stage2 Epoch {epoch+1}: avg_loss = {avg_loss:.4f}')
    print('  Stage 2 complete')


def cleanup_model(model):
    if hasattr(model, 'model') and model.model is not None:
        del model.model
    if hasattr(model, 'tokenizer') and model.tokenizer is not None:
        del model.tokenizer
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print('  GPU memory freed')


def convert_to_serializable(obj):
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


print('Helper functions defined')


Helper functions defined


## Step 8: Seed Configuration

In [11]:
# ============================================================
# SEED FOR THIS NOTEBOOK -- do NOT change
SEED = 42
# ============================================================

RESULT_FILE = OUTPUT_DIR / f'results_seed_{SEED}.json'
PRED_FILE   = OUTPUT_DIR / f'predictions_seed_{SEED}.json'

seed_results = {'seed': SEED, 'rsqwen': None, 'vanilla': None,
                'rsqwen_predictions': None, 'vanilla_predictions': None}

print(f'Seed: {SEED}')
print(f'Results will be saved to: {RESULT_FILE}')


Seed: 42
Results will be saved to: /content/drive/MyDrive/ViTHSD/outputs/results_seed_42.json


## Step 9a: Run RSQwen (Stage 1 + Stage 2)

Saves results immediately after prediction -- safe to crash after this cell.

In [12]:
import time

print('=' * 70)
print(f'RSQwen | seed={SEED}')
print('=' * 70)
t0 = time.time()

set_seed(SEED)
evaluator_rs = Evaluator()
model_rs = create_model(
    'qwen', dataset_type='A',
    batch_size=BATCH_SIZE, num_epochs=2,
    learning_rate=2e-4, lora_r=8, lora_alpha=16
)

print('  Stage 1: Training on Dataset A...')
set_seed(SEED)
model_rs.train(train_texts, train_labels)

print('  Stage 2: Rationale alignment...')
set_seed(SEED)
train_stage2(model_rs, alignment_pairs, r_train_rationale, r_train_labels,
             num_epochs=1, learning_rate=5e-5)

print('  Predicting...')
pred_rs, raw_rs = model_rs.predict(test_texts)
res_rs = evaluator_rs.evaluate(test_labels, pred_rs, 'RSQwen', f'seed_{SEED}')

elapsed = (time.time() - t0) / 60
print(f'  DONE in {elapsed:.1f} min | F1-Micro={res_rs["f1_micro"]:.4f}, F1-Macro={res_rs["f1_macro"]:.4f}')

# Save immediately -- before loading Vanilla model
seed_results['rsqwen'] = convert_to_serializable(res_rs)
seed_results['rsqwen_predictions'] = pred_rs
with open(RESULT_FILE, 'w', encoding='utf-8') as f:
    json.dump(seed_results, f, indent=2, ensure_ascii=False)
with open(PRED_FILE, 'w', encoding='utf-8') as f:
    json.dump(convert_to_serializable({'rsqwen': pred_rs, 'vanilla': None}),
              f, indent=2, ensure_ascii=False)
print(f'  RSQwen results saved to Drive: {RESULT_FILE.name}')

cleanup_model(model_rs)


RSQwen | seed=42
  Seed set to 42
  Stage 1: Training on Dataset A...
  Seed set to 42
[Qwen2.5-3B_MultiLabel] Training with QLoRA (4-bit)...
  Model: Qwen/Qwen2.5-3B-Instruct
  Device: cuda
  Original samples: 7000
  Mode: Standard (Dataset A)
  [Oversampling] Added 540 samples for minority labels
  LoRA config: r=8, alpha=16


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

  Epoch 2: avg_loss = 0.1968
  Training completed
  Stage 2: Rationale alignment...
  Seed set to 42
  Stage 2: 1221 pairs, 1 epoch(s), lr=5e-05


  Stage2 Epoch 1/1: 100%|██████████| 153/153 [03:39<00:00,  1.44s/it, loss=0.4782]


  Stage2 Epoch 1: avg_loss = 0.5475
  Stage 2 complete
  Predicting...


Predicting: 100%|██████████| 1800/1800 [37:29<00:00,  1.25s/it]


  DONE in 77.6 min | F1-Micro=0.5379, F1-Macro=0.3088
  RSQwen results saved to Drive: results_seed_42.json
  GPU memory freed


## Step 9b: Run Vanilla (Stage 1 only)

Loads RSQwen results saved above and appends Vanilla results.

In [13]:
print('=' * 70)
print(f'Vanilla | seed={SEED}')
print('=' * 70)
t0 = time.time()

set_seed(SEED)
evaluator_van = Evaluator()
model_van = create_model(
    'qwen', dataset_type='A',
    batch_size=BATCH_SIZE, num_epochs=2,
    learning_rate=2e-4, lora_r=8, lora_alpha=16
)

print('  Stage 1: Training on Dataset A...')
set_seed(SEED)
model_van.train(train_texts, train_labels)

print('  Predicting...')
pred_van, raw_van = model_van.predict(test_texts)
res_van = evaluator_van.evaluate(test_labels, pred_van, 'Vanilla', f'seed_{SEED}')

elapsed = (time.time() - t0) / 60
print(f'  DONE in {elapsed:.1f} min | F1-Micro={res_van["f1_micro"]:.4f}, F1-Macro={res_van["f1_macro"]:.4f}')

# Load existing partial results and append Vanilla
with open(RESULT_FILE, 'r', encoding='utf-8') as f:
    seed_results = json.load(f)
seed_results['vanilla'] = convert_to_serializable(res_van)
seed_results['vanilla_predictions'] = pred_van
with open(RESULT_FILE, 'w', encoding='utf-8') as f:
    json.dump(seed_results, f, indent=2, ensure_ascii=False)

with open(PRED_FILE, 'r', encoding='utf-8') as f:
    pred_data = json.load(f)
pred_data['vanilla'] = pred_van
with open(PRED_FILE, 'w', encoding='utf-8') as f:
    json.dump(convert_to_serializable(pred_data), f, indent=2, ensure_ascii=False)

print(f'  Vanilla results saved to Drive: {RESULT_FILE.name}')
cleanup_model(model_van)


Vanilla | seed=42
  Seed set to 42
  Stage 1: Training on Dataset A...
  Seed set to 42
[Qwen2.5-3B_MultiLabel] Training with QLoRA (4-bit)...
  Model: Qwen/Qwen2.5-3B-Instruct
  Device: cuda
  Original samples: 7000
  Mode: Standard (Dataset A)
  [Oversampling] Added 540 samples for minority labels
  LoRA config: r=8, alpha=16


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827
  Training for 2 epochs...


Epoch 1/2: 100%|██████████| 943/943 [17:54<00:00,  1.14s/it, loss=0.2024]


  Epoch 1: avg_loss = 0.2779


Epoch 2/2: 100%|██████████| 943/943 [17:54<00:00,  1.14s/it, loss=0.1873]


  Epoch 2: avg_loss = 0.1961
  Training completed
  Predicting...


Predicting: 100%|██████████| 1800/1800 [24:25<00:00,  1.23it/s]


  DONE in 60.4 min | F1-Micro=0.5837, F1-Macro=0.3159
  Vanilla results saved to Drive: results_seed_42.json
  GPU memory freed


## Step 10: Summary Table

In [14]:
with open(RESULT_FILE, 'r', encoding='utf-8') as f:
    final = json.load(f)

print('=' * 60)
print(f'  SEED {SEED} RESULTS SUMMARY')
print('=' * 60)
print(f'{"Metric":>25} {"RSQwen":>12} {"Vanilla":>12}')
print('-' * 50)

metrics = ['f1_micro', 'f1_macro', 'precision_micro', 'recall_micro', 'subset_accuracy']
for m in metrics:
    rs_val = final['rsqwen'][m] if final['rsqwen'] else float('nan')
    va_val = final['vanilla'][m] if final['vanilla'] else float('nan')
    print(f'{m:>25} {rs_val*100:11.2f}% {va_val*100:11.2f}%')

print(f'\nFull results at: {RESULT_FILE}')
print('Run R3_R6_R8_analysis.ipynb after all 3 seeds complete to aggregate.')


  SEED 42 RESULTS SUMMARY
                   Metric       RSQwen      Vanilla
--------------------------------------------------
                 f1_micro       53.79%       58.37%
                 f1_macro       30.88%       31.59%
          precision_micro       48.34%       61.04%
             recall_micro       60.63%       55.92%
          subset_accuracy       43.50%       54.83%

Full results at: /content/drive/MyDrive/ViTHSD/outputs/results_seed_42.json
Run R3_R6_R8_analysis.ipynb after all 3 seeds complete to aggregate.
